In [ ]:
!pip install datasets transformers sentencepiece accelerate -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-uz"
)

print(dataset)

README.md: 0.00B [00:00, ?B/s]

en-uz/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

en-uz/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

en-uz/validation-00000-of-00001.parquet:   0%|          | 0.00/216k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/173157 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 173157
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [ ]:
sample = dataset["train"][0]

print(sample)

{'translation': {'en': "Printer '%s' is out of paper.", 'uz': "'%s' printerda qogʻoz tugadi."}}


In [ ]:
import pandas as pd

train_df = pd.DataFrame(dataset["train"])
valid_df = pd.DataFrame(dataset["validation"])
test_df  = pd.DataFrame(dataset["test"])

In [ ]:
train_df["en"] = train_df["translation"].apply(lambda x: x["en"])
train_df["uz"] = train_df["translation"].apply(lambda x: x["uz"])

train_df = train_df[["en", "uz"]]

train_df.head()

,en,uz
0,Printer '%s' is out of paper.,'%s' printerda qogʻoz tugadi.
1,But surely he who bears patiently and is forgi...,Яхшилар сифати бўлган ушбу сифатларга такрор-т...
2,Intersect Paths,Obʼektlarni guruhlash
3,Lodge them where you lodge according to your m...,"Агар ҳомиладор бўлсалар, то ҳомилаларини қўйгу..."
4,"If He willed, He could still the wind, and the...","Агар У зот хоҳласа, шамолни тўхтатиб қўюр. Бас..."


In [ ]:
import re

def clean_text(text):
    text = str(text)

    # extra spaces
    text = re.sub(r"\s+", " ", text)

    # remove weird unicode chars
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")

    # trim
    text = text.strip()

    return text

In [ ]:
train_df["en"] = train_df["en"].apply(clean_text)
train_df["uz"] = train_df["uz"].apply(clean_text)

/tmp/ipykernel_4139/355028735.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["en"] = train_df["en"].apply(clean_text)
/tmp/ipykernel_4139/355028735.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["uz"] = train_df["uz"].apply(clean_text)


In [ ]:
train_df = train_df[
    (train_df["en"].str.len() > 0) &
    (train_df["uz"].str.len() > 0)
]

In [ ]:
train_df = train_df.drop_duplicates()

In [ ]:
MIN_LEN = 3
MAX_LEN = 300

train_df = train_df[
    (train_df["en"].str.len().between(MIN_LEN, MAX_LEN)) &
    (train_df["uz"].str.len().between(MIN_LEN, MAX_LEN))
]

In [ ]:
def valid_ratio(row):
    en_len = len(row["en"].split())
    uz_len = len(row["uz"].split())

    ratio = max(en_len, uz_len) / max(1, min(en_len, uz_len))

    return ratio < 3

train_df = train_df[
    train_df.apply(valid_ratio, axis=1)
]

In [ ]:
train_df

,en,uz
0,Printer '%s' is out of paper.,'%s' printerda qogʻoz tugadi.
1,But surely he who bears patiently and is forgi...,Яхшилар сифати бўлган ушбу сифатларга такрор-т...
2,Intersect Paths,Obʼektlarni guruhlash
4,"If He willed, He could still the wind, and the...","Агар У зот хоҳласа, шамолни тўхтатиб қўюр. Бас..."
5,"Their sides draw away from (their) beds, they ...",Уларнинг ёнбошлари ётар жойдан йироқ бўлур. Ул...
...,...,...
173152,KFourInLine,UfqGenericName
173153,So spoke those before them as these men say; t...,Улардан олдингилар ҳам уларнинг сўзига ўхшаш г...
173154,"As for those who deny (the Truth), a grievous ...","Кофирларга эса, уларга, шиддатли азоб бордир."
173155,What do you think of the Lord of the Worlds',Оламларнинг Робби ҳақида нима гумонингиз бор?!...


In [ ]:
!pip install cyrtranslit -q

In [ ]:
import cyrtranslit

text = "Агар У зот хоҳласа, шамолни тўхтатиб қўюр."

latin =cyrtranslit.to_latin(text, "uz")

print(latin)

Агар У зот хоҳласа, шамолни тўхтатиб қўюр.


In [ ]:
all_chars = set()

for text in train_df["uz"]:
    all_chars.update(text)

print("Unique chars soni:", len(all_chars))

print(sorted(all_chars))

Unique chars soni: 186
[' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '~', '©', '«', '¬', '®', '±', '»', '×', 'à', 'ä', 'é', 'ö', 'ü', 'ʻ', 'ʼ', 'Ё', 'Ў', 'А', 'Б', 'В', 'Г', 'Д', 'Е', 'Ж', 'З', 'И', 'Й', 'К', 'Л', 'М', 'Н', 'О', 'П', 'Р', 'С', 'Т', 'У', 'Ф', 'Х', 'Ц', 'Ч', 'Ш', 'Э', 'Ю', 'Я', 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ь', 'э', 'ю', 'я', 'ё', 'ў', 'Ғ', 'ғ', 'Қ', 'қ', 'Ҳ', 'ҳ', 'ӯ', '–', '—', '‘', '’', '“', '”', '…', '�']


In [ ]:
import re

REPLACEMENTS = {
    "«": '"',
    "»": '"',

    "ʻ": "'",
    "ʼ": "'",
    "’": "'",
    "‘": "'",
    "`": "'",

    "“": '"',
    "”": '"',

    "–": "-",
    "—": "-",

    "…": "...",

    "�": "",
}

In [ ]:
CYRILLIC_TO_LATIN = {
    'А': 'A', 'а': 'a',
    'Б': 'B', 'б': 'b',
    'В': 'V', 'в': 'v',
    'Г': 'G', 'г': 'g',
    'Ғ': "G'", 'ғ': "g'",
    'Д': 'D', 'д': 'd',
    'Е': 'E', 'е': 'e',
    'Ё': 'Yo', 'ё': 'yo',
    'Ж': 'J', 'ж': 'j',
    'З': 'Z', 'з': 'z',
    'И': 'I', 'и': 'i',
    'Й': 'Y', 'й': 'y',
    'К': 'K', 'к': 'k',
    'Қ': 'Q', 'қ': 'q',
    'Л': 'L', 'л': 'l',
    'М': 'M', 'м': 'm',
    'Н': 'N', 'н': 'n',
    'О': 'O', 'о': 'o',
    'П': 'P', 'п': 'p',
    'Р': 'R', 'р': 'r',
    'С': 'S', 'с': 's',
    'Т': 'T', 'т': 't',
    'У': 'U', 'у': 'u',
    'Ф': 'F', 'ф': 'f',
    'Х': 'X', 'х': 'x',
    'Ҳ': 'H', 'ҳ': 'h',
    'Ц': 'S', 'ц': 's',
    'Ч': 'Ch', 'ч': 'ch',
    'Ш': 'Sh', 'ш': 'sh',
    'Щ': 'Sh', 'щ': 'sh',
    'Ъ': "'", 'ъ': "'",
    'Ь': '', 'ь': '',
    'Э': 'E', 'э': 'e',
    'Ю': 'Yu', 'ю': 'yu',
    'Я': 'Ya', 'я': 'ya',
    'Ў': "O'", 'ў': "o'",
}

In [ ]:
def clean_uzbek_text(text):
    text = str(text)

    # normalize special chars
    for old, new in REPLACEMENTS.items():
        text = text.replace(old, new)

    # cyrillic -> latin
    text = ''.join(
        CYRILLIC_TO_LATIN.get(ch, ch)
        for ch in text
    )

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # trim
    text = text.strip()

    return text

In [ ]:
# text = "Агар У зот хоҳласа, шамолни тўхтатиб қўюр."
# text = "Уларнинг ёнбошлари ётар жойдан йироқ бўлур."
text = "Obʼektlarni guruhlash"

latin =clean_uzbek_text(text)

print(latin)

Ob'ektlarni guruhlash


In [ ]:
train_df["uz"] = train_df["uz"].apply(clean_uzbek_text)

/tmp/ipykernel_4139/2432495404.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["uz"] = train_df["uz"].apply(clean_uzbek_text)


In [ ]:
train_df.head(10)

,en,uz
0,Printer '%s' is out of paper.,'%s' printerda qog'oz tugadi.
1,But surely he who bears patiently and is forgi...,Yaxshilar sifati bo'lgan ushbu sifatlarga takr...
2,Intersect Paths,Ob'ektlarni guruhlash
4,"If He willed, He could still the wind, and the...","Agar U zot xohlasa, shamolni to'xtatib qo'yur...."
5,"Their sides draw away from (their) beds, they ...",Ularning yonboshlari yotar joydan yiroq bo'lur...
6,Who shall inherit Paradise; therein they shall...,"Ular Firdavsni meros olurlar, ular unda abadiy..."
7,Video,Video
8,ARS,ARS
9,And round them shall go boys of theirs as if t...,Va atroflarida xuddi sadafdagi durdek g'ulomla...
10,"Our Lord, and lay not upon us a burden like th...","Bas, kofir qavmlarga bizni g'olib qil. (Hamma ..."


In [ ]:
all_chars = set()

for text in train_df["uz"]:
    all_chars.update(text)

print(sorted(all_chars))

[' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '~', '©', '¬', '®', '±', '×', 'à', 'ä', 'é', 'ö', 'ü', 'ӯ']


In [ ]:
weird_chars = ['à', 'ä', 'é', 'ö', 'ü', "ӯ"]

for ch in weird_chars:
    count = train_df["uz"].str.contains(ch, regex=False).sum()
    print(ch, count)

à 6
ä 1
é 6
ö 4
ü 4
ӯ 1


In [ ]:
train_df["en_len"] = train_df["en"].str.len()
train_df["uz_len"] = train_df["uz"].str.len()

train_df["total_len"] = train_df["en_len"] + train_df["uz_len"]

/tmp/ipykernel_4139/1596254469.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["en_len"] = train_df["en"].str.len()
/tmp/ipykernel_4139/1596254469.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["uz_len"] = train_df["uz"].str.len()
/tmp/ipykernel_4139/1596254469.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pand

In [ ]:
df = train_df.sort_values(
    by="total_len",
    ascending=True
)

In [ ]:
df[['en', 'uz']].to_csv(
    "opus_en_uz_clean_sorted.csv",
    index=False,
    encoding="utf-8"
)